In [57]:
"""
    File: modello.ipynb
    Author: Andrea Macale
    Date: 2026-03-04

    Description: Notebook per la realizzazione del modello per il suggerimento ed analisi di follow-up clinici

"""

'\n    File: modello.ipynb\n    Author: Andrea Macale\n    Date: 2026-03-04\n\n    Description: Notebook per la realizzazione del modello per il suggerimento ed analisi di follow-up clinici\n\n'

# Parte 0: Importazione delle librerie

## Librerie principali

In [58]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

## Correlazione (Spearman, Pearson)

In [59]:
import scipy.stats
from scipy.stats import pearsonr, spearmanr

## VIF

In [60]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import statsmodels.api as sm

## Modelli di ML

# Parte 1: Estrazione dei dati

## Aperta dei file del dataset

In [61]:
pos_dataset = os.path.join(Path(os.path.abspath("modello.ipynb")).parent.parent, "data")

In [62]:
file = os.path.join(pos_dataset, "patients.csv")
pazienti = pd.read_csv(file)
pazienti['subject_id'] = pazienti['subject_id'].astype(str)
pazienti.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [63]:
file = os.path.join(pos_dataset, "diagnoses_icd.csv")
tipi = {
    'subject_id': str, 
    'hadm_id': str, 
    'icd_code': str, 
    'icd_version': str
}
diagnosi = pd.read_csv(file, dtype=tipi)
diagnosi['icd_version'] = pd.to_numeric(diagnosi['icd_version'], errors='coerce').fillna(0).astype(int)
diagnosi = diagnosi.drop_duplicates()
diagnosi.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


In [64]:
file = os.path.join(pos_dataset, "mimic-cxr-2.0.0-metadata.csv")
metadati = pd.read_csv(file)
metadati['dicom_id'] = metadati['dicom_id'].astype(str)
metadati['subject_id'] = metadati['subject_id'].astype(str)
metadati['study_id'] = metadati['study_id'].astype(str)
metadati.head()

,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,CHEST (PA AND LAT),PA,3056,2544,21800506,213014.531,CHEST (PA AND LAT),postero-anterior,Erect
1,174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962,10000032,50414267,CHEST (PA AND LAT),LATERAL,3056,2544,21800506,213014.531,CHEST (PA AND LAT),lateral,Erect
2,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,CHEST (PA AND LAT),PA,3056,2544,21800626,165500.312,CHEST (PA AND LAT),postero-anterior,Erect
3,e084de3b-be89b11e-20fe3f9f-9c8d8dfe-4cfd202c,10000032,53189527,CHEST (PA AND LAT),LATERAL,3056,2544,21800626,165500.312,CHEST (PA AND LAT),lateral,Erect
4,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,CHEST (PORTABLE AP),AP,2705,2539,21800723,80556.875,CHEST (PORTABLE AP),antero-posterior,NaN


In [65]:
lista_referti = []
ricerca = Path(os.path.join(pos_dataset, "mimic-cxr-reports", "files"))
for file_path in ricerca.rglob('*.txt'):
    study_id = file_path.stem.replace('s', '')
    subject_id = file_path.parent.name.replace('p', '')
    with open(file_path, 'r', encoding='utf-8') as file:
        testo = file.read()
    lista_referti.append({'subject_id': subject_id, 'study_id': study_id, 'testo_referto': testo})
referti = pd.DataFrame(lista_referti)
print(f"{len(referti)} referti")
referti.head()

227835 referti


,subject_id,study_id,testo_referto
0,17015391,58847767,FINAL REPORT\...
1,17015391,51918981,FINAL REPORT\...
2,17015391,54239234,FINAL REPORT\...
3,17015391,51167371,FINAL REPORT\...
4,17015391,52622416,FINAL REPORT\...


In [66]:
lista_immagini = []
ricerca = Path(os.path.join(pos_dataset, "MIMIC_SUPER_RES_24K"))
for file_path in ricerca.rglob('*.jpg'):
    dicom_id = file_path.stem
    lista_immagini.append({'dicom_id': dicom_id, 'path_immagine': str(file_path)})
immagini = pd.DataFrame(lista_immagini)
immagini['path_immagine'] = immagini['path_immagine'].str.replace(str(pos_dataset+"/"), "data/", regex=False) # pulisci il path
immagini['dicom_id'] = immagini['dicom_id'].astype(str)
print(f"{len(immagini)} immagini")
immagini.head()

24000 immagini


,dicom_id,path_immagine
0,62456122-1887ff8c-ef1cba37-270fe1ac-c88ef7fc,data/MIMIC_SUPER_RES_24K/62456122-1887ff8c-ef1...
1,152c3005-d0c7bf7a-4c0f7935-e82aef6e-bed96e2e,data/MIMIC_SUPER_RES_24K/152c3005-d0c7bf7a-4c0...
2,c4a20cd5-94e7d182-0bbca8a9-797c8bf5-7e12e050,data/MIMIC_SUPER_RES_24K/c4a20cd5-94e7d182-0bb...
3,064741c9-95255009-ae0c53ee-ccb54164-21b77bbb,data/MIMIC_SUPER_RES_24K/064741c9-95255009-ae0...
4,2150ba84-0f87ae6f-a7132d8e-8b02fb6c-99745702,data/MIMIC_SUPER_RES_24K/2150ba84-0f87ae6f-a71...


## Prima pulizia, selezione e join dei dati

In [67]:
dataset = pazienti.merge(metadati, on=['subject_id'], how='inner')
dataset = dataset.merge(immagini, on=['dicom_id'], how='inner')
dataset = dataset.merge(referti, on=['subject_id', 'study_id'], how='inner')
print(len(dataset))
dataset

22768


,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod,dicom_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning,path_immagine,testo_referto
0,10000898,F,80,2188,2014 - 2016,NaN,2a280266-c8bae121-54d75383-cac046f4-ca37aa16,50771383,CHEST (PA AND LAT),PA,2544,3056,21880312,125501.842,CHEST (PA AND LAT),postero-anterior,Erect,data/MIMIC_SUPER_RES_24K/2a280266-c8bae121-54d...,FINAL REPORT\...
1,10000935,F,52,2182,2008 - 2010,2187-11-12,f1adcae3-2921c0a8-5d9652f9-4191ecd7-f2a96f35,56522600,NaN,PA,2022,1736,21860730,155005.000,CHEST (PA AND LAT),postero-anterior,Erect,data/MIMIC_SUPER_RES_24K/f1adcae3-2921c0a8-5d9...,FINAL REPORT\...
2,10001401,F,89,2131,2014 - 2016,NaN,0009a9fb-eb905e90-824cad7c-16d40468-007f0038,50225296,PORTABLE ABDOMEN,AP,3056,2544,21310610,102234.484,DX CHEST PORT LINE/TUBE PLCMT 1 EXAM,antero-posterior,Erect,data/MIMIC_SUPER_RES_24K/0009a9fb-eb905e90-824...,FINAL REPORT\...
3,10001401,F,89,2131,2014 - 2016,NaN,c74ce171-c7c53262-a7d57fa1-ee9a9bea-b5f75cb8,51065211,CHEST (PA AND LAT),PA,3056,2544,21310619,193047.156,CHEST (PA AND LAT),postero-anterior,Erect,data/MIMIC_SUPER_RES_24K/c74ce171-c7c53262-a7d...,FINAL REPORT\...
4,10001401,F,89,2131,2014 - 2016,NaN,d9db838d-4612fd1e-e45b40a9-3ea30033-26efd8e4,55350604,Performed Desc,PA,2021,2021,21310802,114559.000,CHEST (PA AND LAT),postero-anterior,Recumbent,data/MIMIC_SUPER_RES_24K/d9db838d-4612fd1e-e45...,FINAL REPORT\...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22763,19998330,F,71,2177,2011 - 2013,2178-12-08,05405a03-c11efc6e-8a4366b6-d929640f-c356d564,59735820,CHEST (PORTABLE AP),AP,2453,2306,21781021,170349.875,CHEST (PORTABLE AP),antero-posterior,Erect,data/MIMIC_SUPER_RES_24K/05405a03-c11efc6e-8a4...,FINAL REPORT\...
22764,19998350,M,52,2127,2011 - 2013,NaN,d14b91a7-2deb65ba-dea8e4b9-b7bbeb5e-f3ac734f,51819111,CHEST (PA AND LAT),PA,3056,2544,21280221,85427.984,CHEST (PA AND LAT),postero-anterior,Erect,data/MIMIC_SUPER_RES_24K/d14b91a7-2deb65ba-dea...,FINAL REPORT\...
22765,19999287,F,71,2191,2008 - 2010,2197-09-02,50373d1b-8df0d15f-5d0047f4-578bc509-4f2b48f0,51885769,CHEST (PORTABLE AP),AP,3056,2544,21970726,25712.640,CHEST (PORTABLE AP),antero-posterior,Erect,data/MIMIC_SUPER_RES_24K/50373d1b-8df0d15f-5d0...,FINAL REPORT\...
22766,19999287,F,71,2191,2008 - 2010,2197-09-02,46510b80-411ac511-fe6ffab2-d7dfdc76-dff1a762,52519175,CHEST (PORTABLE AP),AP,2544,3056,21970807,90637.796,CHEST (PORTABLE AP),antero-posterior,NaN,data/MIMIC_SUPER_RES_24K/46510b80-411ac511-fe6...,FINAL REPORT\...


In [68]:
condizioni = [
    # STADIO IV: Metastasi (Il più grave)
    diagnosi['icd_code'].str.startswith(('196', '197', '198', '199', 'C77', 'C78', 'C79'), na=False),
    
    # STADIO I-III: Tumore Primario Invasivo
    diagnosi['icd_code'].str.startswith(('162', 'C34'), na=False),
    
    # BENIGNO: Tumori non maligni (ICD-9: 212.3 | ICD-10: D14.3)
    diagnosi['icd_code'].str.startswith(('2123', 'D143'), na=False),
    
    # A RISCHIO: Noduli o ombre (Sospetti non confermati)
    diagnosi['icd_code'].str.contains('793.1|R91', na=False)
]
valori = [4, 3, 2, 1]
diagnosi['numero_severita'] = np.select(condizioni, valori, default=0)
etichette_pazienti = diagnosi.groupby('subject_id')['numero_severita'].max().reset_index()
mappa_severita = {
    4: 'IV STADIO: METASTATICO',
    3: 'I-III STADIO: PRIMARIO',
    2: 'BENIGNO',
    1: 'A RISCHIO',
    0: 'NEGATIVO'
}
etichette_pazienti['stato_clinico'] = etichette_pazienti['numero_severita'].map(mappa_severita)

In [69]:
dataset = dataset.merge(etichette_pazienti, on=['subject_id'], how='left')
dataset['stato_clinico'] = dataset['stato_clinico'].fillna('NEGATIVO')
dataset

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod,dicom_id,study_id,PerformedProcedureStepDescription,ViewPosition,...,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning,path_immagine,testo_referto,numero_severita,stato_clinico
0,10000898,F,80,2188,2014 - 2016,NaN,2a280266-c8bae121-54d75383-cac046f4-ca37aa16,50771383,CHEST (PA AND LAT),PA,...,3056,21880312,125501.842,CHEST (PA AND LAT),postero-anterior,Erect,data/MIMIC_SUPER_RES_24K/2a280266-c8bae121-54d...,FINAL REPORT\...,NaN,NEGATIVO
1,10000935,F,52,2182,2008 - 2010,2187-11-12,f1adcae3-2921c0a8-5d9652f9-4191ecd7-f2a96f35,56522600,NaN,PA,...,1736,21860730,155005.000,CHEST (PA AND LAT),postero-anterior,Erect,data/MIMIC_SUPER_RES_24K/f1adcae3-2921c0a8-5d9...,FINAL REPORT\...,4.0,IV STADIO: METASTATICO
2,10001401,F,89,2131,2014 - 2016,NaN,0009a9fb-eb905e90-824cad7c-16d40468-007f0038,50225296,PORTABLE ABDOMEN,AP,...,2544,21310610,102234.484,DX CHEST PORT LINE/TUBE PLCMT 1 EXAM,antero-posterior,Erect,data/MIMIC_SUPER_RES_24K/0009a9fb-eb905e90-824...,FINAL REPORT\...,1.0,A RISCHIO
3,10001401,F,89,2131,2014 - 2016,NaN,c74ce171-c7c53262-a7d57fa1-ee9a9bea-b5f75cb8,51065211,CHEST (PA AND LAT),PA,...,2544,21310619,193047.156,CHEST (PA AND LAT),postero-anterior,Erect,data/MIMIC_SUPER_RES_24K/c74ce171-c7c53262-a7d...,FINAL REPORT\...,1.0,A RISCHIO
4,10001401,F,89,2131,2014 - 2016,NaN,d9db838d-4612fd1e-e45b40a9-3ea30033-26efd8e4,55350604,Performed Desc,PA,...,2021,21310802,114559.000,CHEST (PA AND LAT),postero-anterior,Recumbent,data/MIMIC_SUPER_RES_24K/d9db838d-4612fd1e-e45...,FINAL REPORT\...,1.0,A RISCHIO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22763,19998330,F,71,2177,2011 - 2013,2178-12-08,05405a03-c11efc6e-8a4366b6-d929640f-c356d564,59735820,CHEST (PORTABLE AP),AP,...,2306,21781021,170349.875,CHEST (PORTABLE AP),antero-posterior,Erect,data/MIMIC_SUPER_RES_24K/05405a03-c11efc6e-8a4...,FINAL REPORT\...,0.0,NEGATIVO
22764,19998350,M,52,2127,2011 - 2013,NaN,d14b91a7-2deb65ba-dea8e4b9-b7bbeb5e-f3ac734f,51819111,CHEST (PA AND LAT),PA,...,2544,21280221,85427.984,CHEST (PA AND LAT),postero-anterior,Erect,data/MIMIC_SUPER_RES_24K/d14b91a7-2deb65ba-dea...,FINAL REPORT\...,0.0,NEGATIVO
22765,19999287,F,71,2191,2008 - 2010,2197-09-02,50373d1b-8df0d15f-5d0047f4-578bc509-4f2b48f0,51885769,CHEST (PORTABLE AP),AP,...,2544,21970726,25712.640,CHEST (PORTABLE AP),antero-posterior,Erect,data/MIMIC_SUPER_RES_24K/50373d1b-8df0d15f-5d0...,FINAL REPORT\...,3.0,I-III STADIO: PRIMARIO
22766,19999287,F,71,2191,2008 - 2010,2197-09-02,46510b80-411ac511-fe6ffab2-d7dfdc76-dff1a762,52519175,CHEST (PORTABLE AP),AP,...,3056,21970807,90637.796,CHEST (PORTABLE AP),antero-posterior,NaN,data/MIMIC_SUPER_RES_24K/46510b80-411ac511-fe6...,FINAL REPORT\...,3.0,I-III STADIO: PRIMARIO
